# LIBERO-Plus, family 4: Diffusion Policy (trained here, LeRobot)
**GPU: A100** (training). Steps: pick the LIBERO-10 episodes of `lerobot/libero`, train 60k steps (about 4 to 6 h; checkpoints
mirrored to Drive every 5k, so a disconnect resumes), then swap in LIBERO-Plus and evaluate on the shared subset (~1 h).
Diffusion Policy has no language input, so it is expected to be the weakest family; that is fine, it is the non-VLA baseline.
Paste back the episode-selection line, the final training lines, and the last table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline matplotlib backend crashes every venv python
os.environ['PYTHONUNBUFFERED'] = '1'   # run cells are piped into a log on Drive; without this their output arrives in bursts
os.makedirs('/content/drive/MyDrive/restoring_organs/plus/logs', exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)
print('outputs ->', '/content/drive/MyDrive/restoring_organs/plus')
# Drive uploads a file only after it is CLOSED, so a log held open by a running cell never arrives. Run cells log to
# /content/logs; this thread rewrites the last 64 KB of each log on Drive once a minute as a fresh, closed file.
import builtins, glob, threading, time
def _mirror_logs():
    while True:
        for p in glob.glob('/content/logs/*.log'):
            try:
                with open(p, 'rb') as f:
                    f.seek(max(0, os.path.getsize(p) - 65536))
                    tail = f.read()
                with open('/content/drive/MyDrive/restoring_organs/plus/logs/' + os.path.basename(p)[:-4] + '.' + os.uname().nodename[-6:] + '.log', 'wb') as f:
                    f.write(tail)
            except Exception:
                pass
        time.sleep(60)
if not getattr(builtins, '_log_mirror_on', False):
    builtins._log_mirror_on = True
    threading.Thread(target=_mirror_logs, daemon=True).start()
# Hugging Face login from the Colab secret HF_TOKEN (key icon, left sidebar), if you added one; optional here
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('Hugging Face login ok')
except Exception as e:
    print('no HF_TOKEN secret, continuing without Hugging Face login:', type(e).__name__)

In [ ]:
import os; os.makedirs('/content/scripts', exist_ok=True); print('ok')

In [ ]:
%%writefile /content/scripts/plus_common.py
"""Shared by every LIBERO-Plus notebook: one deterministic task subset (so all five policy families see
exactly the same perturbed tasks), perturbation categories parsed from task names, and the summary table."""
import json
import math
import os
import random
import re
from collections import Counter, defaultdict

# LIBERO-Plus task names, checked on the real suite: token perturbations carry their own tag; camera and robot
# initial state are encoded as values, e.g. ..._view_0_0_100_0_0_initstate_0 is the unperturbed setting.
CATS = [("sensor_noise", r"_noise_\d+"), ("language", r"_language_\d+"), ("light", r"_light_\d+"),
        ("background", r"_table_\d+|_tb_\d+"), ("layout", r"_add_\d+|_level")]
DEFAULT_VIEW = ("0", "0", "100", "0", "0")
# Every run script reads the suite from the environment, so one variable switches a whole notebook:
#   LIBERO_SUITE=libero_spatial | libero_object | libero_goal | libero_10 (default; nothing changes when it is unset)
SUITE = os.environ.get("LIBERO_SUITE", "libero_10")
MAX_STEPS = {"libero_spatial": 220, "libero_object": 280, "libero_goal": 300, "libero_10": 520}     # OFT's own horizons
OFT_CKPT = {s: "moojink/openvla-7b-oft-finetuned-" + s.replace("_", "-") for s in MAX_STEPS}


def categorize(name):
    for cat, pat in CATS:
        if re.search(pat, name):
            return cat
    m = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", name)
    if m and m.groups() != DEFAULT_VIEW:
        return "camera"
    s = re.search(r"_initstate_(\d+)", name)
    if s and s.group(1) != "0":
        return "robot_init"
    return "other"


def build_subset(suite_name="libero_10", per_cat=30, seed=0):
    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()[suite_name]()
    names = [suite.get_task(i).name for i in range(suite.n_tasks)]
    langs = [suite.get_task(i).language for i in range(suite.n_tasks)]
    by = defaultdict(list)
    for i, n in enumerate(names):
        by[categorize(n)].append(i)
    rng = random.Random(seed)
    pick = []
    for cat in sorted(by):
        ids = sorted(by[cat])
        rng.shuffle(ids)
        pick += ids[:per_cat]
    pick.sort()
    tokens = Counter(tok for n in names for tok in re.findall(r"_([a-z]+)_\d+", n))
    return {"suite": suite_name, "n_tasks": suite.n_tasks, "per_cat": per_cat, "seed": seed,
            "counts": {c: len(v) for c, v in sorted(by.items())},
            "other_examples": [names[i] for i in by.get("other", [])[:5]],
            "name_tokens": dict(tokens.most_common(20)),
            "tasks": [{"task_id": i, "name": names[i], "language": langs[i], "category": categorize(names[i])}
                      for i in pick]}


def load_or_build(path, per_cat=30):
    if os.path.exists(path):
        return json.load(open(path))
    sub = build_subset(suite_name=SUITE, per_cat=per_cat)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    json.dump(sub, open(path, "w"), indent=1)
    return sub


def load_rows(path):
    rows = []
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows


def append_row(path, row):
    """Append one result row and CLOSE the file. Drive uploads a file only when it is closed: a runner that keeps its
    results file open for a whole cell has nothing on Drive until the cell ends, and a killed runtime loses all of it."""
    with open(path, "a") as f:
        f.write(json.dumps(row, default=float) + "\n")


def load_shards(path):
    """Rows of `<stem>.jsonl` plus every per-session shard `<stem>.<shard>.jsonl` next to it, one row per
    (task, condition, arm, rep). Two Colab sessions that append to ONE Drive file overwrite each other (each uploads its
    own whole copy, the last upload wins; file 8 lost 576 rows that way on 2026-09-20), so every session writes its own
    shard and readers merge. A measured row beats one recovered from a log (`recovered`), which lacks the detail fields."""
    import glob
    best = {}
    # 0 = measured, 1 = measured before the shading gate (re-queued by the runner, its re-run lives in a later shard),
    # 2 = rebuilt from a log. The lowest rank wins; on a tie the first file does.
    rank = lambda r: 2 if r.get("recovered") else int(bool(r.get("photo_applied")) and "photo_shading" not in r)
    for p in [path] + sorted(glob.glob(path[:-len(".jsonl")] + ".*.jsonl")):
        for r in load_rows(p):
            k = (r.get("policy"), r["task_id"], r.get("cond"), r["arm"], r.get("rep", 0))
            if k not in best or rank(r) < rank(best[k]):
                best[k] = r
    return list(best.values())


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)


def summarize(rows, label=""):
    by = defaultdict(list)
    for r in rows:
        by[r["category"]].append(bool(r["success"]))
    out = {}
    print(f"\n{label}  ({len(rows)} episodes)")
    print(f"{'category':<14}{'n':>5}{'success':>9}{'95% CI':>16}")
    for cat in sorted(by) + ["ALL"]:
        v = [s for r in by.values() for s in r] if cat == "ALL" else by[cat]
        k, n = sum(v), len(v)
        lo, hi = wilson(k, n)
        out[cat] = {"n": n, "success": k / max(1, n), "ci": [lo, hi]}
        print(f"{cat:<14}{n:>5}{k / max(1, n):>9.3f}   [{lo:.2f}, {hi:.2f}]")
    return out

In [ ]:
%%writefile /content/scripts/lerobot_tools.py
"""LeRobot side: resumable lerobot-eval batches over the LIBERO-Plus subset (pi0, pi0.5, Diffusion Policy),
result parsing, and selection of the LIBERO-10 episodes inside lerobot/libero for Diffusion Policy training."""
import argparse
import glob
import json
import os
import re
import shutil
import subprocess
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import summarize  # noqa: E402


def _walk(x):
    if isinstance(x, dict):
        yield x
        for v in x.values():
            yield from _walk(v)
    elif isinstance(x, list):
        for v in x:
            yield from _walk(v)


def parse_eval_info(path):
    """{task_id: success}. Handles the per_task layout of eval_policy_all and falls back to any dict
    carrying a task_id and metrics."""
    info = json.load(open(path))
    out = {}
    for d in _walk(info):
        if "task_id" in d and isinstance(d.get("metrics"), dict):
            m = d["metrics"]
            if "successes" in m and isinstance(m["successes"], list) and m["successes"]:
                ok = any(bool(s) for s in m["successes"])
            elif "pc_success" in m:
                ok = float(m["pc_success"]) > 50.0
            else:
                continue
            out[int(d["task_id"])] = ok
    return out


def run_eval(a):
    sub = json.load(open(a.subset))
    tasks = sub["tasks"][:a.limit] if a.limit else sub["tasks"]
    by_id = {t["task_id"]: t for t in tasks}
    ids = [t["task_id"] for t in tasks]
    os.makedirs(a.out, exist_ok=True)
    for b in range(0, len(ids), a.batch):
        chunk = ids[b:b + a.batch]
        keep = os.path.join(a.out, f"batch_{b // a.batch:03d}.json")
        if os.path.exists(keep):
            continue
        od = f"/content/lerobot_eval/{a.name}/batch_{b // a.batch:03d}"
        if os.path.isdir(od):
            shutil.rmtree(od)
        cmd = ["lerobot-eval", f"--policy.path={a.policy}", "--env.type=libero", "--env.task=libero_10",
               "--env.is_libero_plus=true", "--env.task_ids=[" + ",".join(map(str, chunk)) + "]",
               "--eval.batch_size=1", "--eval.n_episodes=1", "--env.max_parallel_tasks=1", f"--output_dir={od}"]
        if "/pi0" in a.policy:
            cmd.append("--policy.compile_model=false")   # ~11 calls per episode: compiling costs more than it saves
        print(" ".join(cmd), flush=True)
        subprocess.run(cmd, check=True, env={**os.environ, "MUJOCO_GL": "egl"})
        found = glob.glob(f"{od}/**/eval_info.json", recursive=True)
        assert found, f"no eval_info.json under {od}"
        res = parse_eval_info(found[0])
        missing = [i for i in chunk if i not in res]
        assert not missing, f"eval_info.json lacks task ids {missing[:5]}; keys seen {sorted(res)[:5]}"
        json.dump({str(i): res[i] for i in chunk}, open(keep, "w"))
        print(f"batch {b // a.batch}: {sum(res[i] for i in chunk)}/{len(chunk)} success", flush=True)
    rows = []
    for p in sorted(glob.glob(os.path.join(a.out, "batch_*.json"))):
        for k, v in json.load(open(p)).items():
            if int(k) in by_id:
                rows.append(dict(family=a.name, **by_id[int(k)], success=bool(v)))
    with open(os.path.join(a.out, "rows.jsonl"), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    summarize(rows, a.name)


def _norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower().rstrip("."))


def select_episodes(a):
    """Episodes of lerobot/libero whose task string is one of the 10 LIBERO-10 instructions."""
    import pandas as pd
    from huggingface_hub import snapshot_download
    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    langs = {_norm(suite.get_task(i).language) for i in range(suite.n_tasks)}
    assert len(langs) == 10, f"expected 10 LIBERO-10 instructions, got {len(langs)}"
    root = snapshot_download("lerobot/libero", repo_type="dataset", allow_patterns=["meta/*", "meta/**/*"])
    ep_files = sorted(glob.glob(os.path.join(root, "meta", "episodes", "**", "*.parquet"), recursive=True))
    assert ep_files, f"no episode parquet under {root}/meta/episodes"
    eps = pd.concat([pd.read_parquet(p) for p in ep_files], ignore_index=True)
    task_col = "tasks" if "tasks" in eps.columns else None
    if task_col is None:
        tf = glob.glob(os.path.join(root, "meta", "tasks.parquet"))
        assert tf and "task_index" in eps.columns, f"episode columns: {list(eps.columns)}"
        tt = pd.read_parquet(tf[0]).reset_index()
        tcol = [c for c in tt.columns if tt[c].dtype == object][0]
        imap = dict(zip(tt["task_index"], tt[tcol]))
        eps["tasks"] = eps["task_index"].map(lambda i: [imap[i]])
        task_col = "tasks"

    def ep_langs(v):
        return [_norm(x) for x in (list(v) if not isinstance(v, str) else [v])]

    chosen = sorted(int(r["episode_index"]) for _, r in eps.iterrows() if any(l in langs for l in ep_langs(r[task_col])))
    matched = {l for _, r in eps.iterrows() for l in ep_langs(r[task_col]) if l in langs}
    print(f"lerobot/libero: {len(eps)} episodes, {len(chosen)} are LIBERO-10, instructions matched {len(matched)}/10")
    assert len(matched) == 10 and len(chosen) >= 300, "LIBERO-10 episode match failed; paste this output"
    json.dump({"episodes": chosen, "instructions": sorted(langs)}, open(a.out, "w"))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["eval", "select"])
    ap.add_argument("--subset")
    ap.add_argument("--out", required=True)
    ap.add_argument("--policy")
    ap.add_argument("--name")
    ap.add_argument("--batch", type=int, default=30)
    ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()
    run_eval(a) if a.mode == "eval" else select_episodes(a)

In [ ]:
%%bash
set -e

pip -q install "lerobot[libero,diffusion] @ git+https://github.com/huggingface/lerobot.git@v0.6.1"
if ! python -c "import dataclasses; from lerobot.envs.configs import LiberoEnv; assert 'is_libero_plus' in {f.name for f in dataclasses.fields(LiberoEnv)}"; then
  echo "v0.6.1 lacks is_libero_plus, installing lerobot main"
  pip -q install "lerobot[libero,diffusion] @ git+https://github.com/huggingface/lerobot.git"
fi
python -c "import lerobot; print('lerobot', lerobot.__version__ if hasattr(lerobot, '__version__') else 'ok')"

In [ ]:
# LIBERO-10 episodes (uses the stock LIBERO package, before the LIBERO-Plus swap)
import os
if not os.path.exists('/content/drive/MyDrive/restoring_organs/plus/dp_libero10_episodes.json'):
    !python /content/scripts/lerobot_tools.py select --out /content/drive/MyDrive/restoring_organs/plus/dp_libero10_episodes.json
else:
    print('episodes already selected')

In [ ]:
# ---- train Diffusion Policy on the LIBERO-10 episodes of lerobot/libero (resumable via Drive) ----
import glob, json, os, shutil, subprocess, threading, time
STEPS = 60000
LOCAL, DRV = '/content/dp_out', '/content/drive/MyDrive/restoring_organs/plus/dp_libero10_ckpts'
os.makedirs(DRV, exist_ok=True)
eps = json.load(open('/content/drive/MyDrive/restoring_organs/plus/dp_libero10_episodes.json'))['episodes']
steps_on_drive = sorted(int(d) for d in os.listdir(DRV) if d.isdigit())
if steps_on_drive and steps_on_drive[-1] >= STEPS:
    print('training already finished at step', steps_on_drive[-1])
else:
    base = ['lerobot-train', '--policy.type=diffusion', '--policy.push_to_hub=false', '--policy.crop_shape=[224,224]',
            '--policy.horizon=16', '--policy.n_action_steps=8', '--dataset.repo_id=lerobot/libero',
            '--dataset.episodes=[' + ','.join(map(str, eps)) + ']', '--batch_size=64', f'--steps={STEPS}',
            '--save_freq=5000', '--num_workers=8', '--wandb.enable=false', '--job_name=dp_libero10', f'--output_dir={LOCAL}']
    if steps_on_drive:
        last = steps_on_drive[-1]
        src = f'{DRV}/{last}'
        name = [d for d in os.listdir(src)][0] if len(os.listdir(src)) == 1 and not os.path.isdir(f'{src}/pretrained_model') else None
        shutil.rmtree(LOCAL, ignore_errors=True)
        ck = f'{LOCAL}/checkpoints'
        os.makedirs(ck, exist_ok=True)
        shutil.copytree(src if name is None else f'{src}/{name}', f'{ck}/{name or str(last)}')
        os.symlink(f'{ck}/{name or str(last)}', f'{ck}/last')
        cmd = ['lerobot-train', '--resume=true', f'--config_path={ck}/last/pretrained_model/train_config.json']
        print('resuming from step', last)
    else:
        shutil.rmtree(LOCAL, ignore_errors=True)
        cmd = base
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    def mirror():
        while proc.poll() is None:
            time.sleep(120)
            for d in glob.glob(f'{LOCAL}/checkpoints/*'):
                n = os.path.basename(d)
                if n.isdigit() and not os.path.exists(f'{DRV}/{int(n)}') and os.path.exists(f'{d}/pretrained_model') and os.path.exists(f'{d}/training_state'):
                    time.sleep(30)
                    shutil.copytree(d, f'{DRV}/{int(n)}/{n}')
                    print('mirrored checkpoint', n, 'to Drive', flush=True)
    th = threading.Thread(target=mirror, daemon=True); th.start()
    for line in proc.stdout:
        if 'step' in line or 'Error' in line or 'error' in line:
            print(line.rstrip(), flush=True)
    proc.wait(); th.join(timeout=5)
    for d in glob.glob(f'{LOCAL}/checkpoints/*'):
        n = os.path.basename(d)
        if n.isdigit() and not os.path.exists(f'{DRV}/{int(n)}'):
            shutil.copytree(d, f'{DRV}/{int(n)}/{n}')
    assert proc.returncode == 0, 'training failed, paste the output above'
print('checkpoints on Drive:', sorted(int(d) for d in os.listdir(DRV) if d.isdigit()))

In [ ]:
%%bash
set -e
pip -q uninstall -y hf-libero || true

apt-get -qq update > /dev/null
apt-get -qq install -y libexpat1 libfontconfig1-dev libmagickwand-dev unzip > /dev/null
cd /content
[ -d LIBERO-plus ] || git clone -q --depth 1 https://github.com/sylvestf/LIBERO-plus.git
pip -q install -e /content/LIBERO-plus --no-deps
pip -q install wand scikit-image
# LIBERO's setup.py does not register the top-level package, so the editable install alone is not importable
SITE=$(python -c "import sysconfig; print(sysconfig.get_paths()['purelib'])")
echo /content/LIBERO-plus > "$SITE/libero_plus.pth"
python -c "import importlib.util as u; assert u.find_spec('libero') is not None, 'libero still not importable'; print('libero importable')"
if [ ! -f /content/LIBERO-plus/.assets_ok ]; then
  echo "downloading LIBERO-Plus assets (6.4 GB)..."
  # Hugging Face answers a rate-limited or interrupted download with an error page or a short file; plain `curl -s`
  # saved that as the zip and unzip then failed with no message (2026-09-19). Fail on HTTP errors, resume, retry, send
  # the HF_TOKEN when there is one (anonymous downloads are limited first), and test the archive before unpacking.
  ZIP=/content/lp_assets.zip
  URL=https://huggingface.co/datasets/sii-research/LIBERO_plus_assets/resolve/main/assets.zip
  ok=0
  for try in 1 2 3 4 5 6; do
    if [ -f $ZIP ] && unzip -tq $ZIP > /dev/null 2>&1; then ok=1; break; fi
    size=$(stat -c %s $ZIP 2>/dev/null || echo 0)
    if [ $try -gt 1 ]; then echo "asset download attempt $((try-1)) incomplete ($size bytes), retrying in 30 s"; sleep 30; fi
    if [ "$size" -lt 1000000 ]; then rm -f $ZIP; fi                 # an error page cannot be resumed
    curl -fL -C - --retry 3 --retry-delay 15 -sS ${HF_TOKEN:+-H "Authorization: Bearer $HF_TOKEN"} -o $ZIP $URL || true
  done
  if [ $ok != 1 ]; then echo "LIBERO-Plus assets could not be downloaded (Hugging Face rate limit or outage). Wait a few minutes and re-run this cell; it resumes."; exit 1; fi
  rm -rf /content/lp_assets && mkdir -p /content/lp_assets
  unzip -q /content/lp_assets.zip -d /content/lp_assets && rm -f /content/lp_assets.zip
  python3 - <<'EOF'
import os, shutil
root, dst = "/content/lp_assets", "/content/LIBERO-plus/libero/libero/assets"
cands = [os.path.join(d, n) for d, ns, _ in os.walk(root) for n in ns if n == "assets"]
src = min(cands, key=len) if cands else root
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("assets:", src, "->", dst, "|", len(os.listdir(dst)), "entries")
EOF
  touch /content/LIBERO-plus/.assets_ok
fi
python /content/scripts/robosuite_np2.py

In [ ]:
# LIBERO-Plus asks for its paths on first import; write them up front
import os
root = '/content/LIBERO-plus/libero/libero'
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
open(os.path.expanduser('~/.libero/config.yaml'), 'w').write(
    f'benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n'
    f'datasets: {root}/../datasets\nassets: {root}/assets\n')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline backend is not valid inside the venvs
os.environ['PYTHONPATH'] = '/content/LIBERO-plus'   # Colab's system python ignores the .pth file
print(open(os.path.expanduser('~/.libero/config.yaml')).read())

In [ ]:
# the SAME task subset for every family (built once, then read from Drive)
!MUJOCO_GL=egl python -c "import sys; sys.path.insert(0, '/content/scripts'); import plus_common as pc; s = pc.load_or_build('/content/drive/MyDrive/restoring_organs/plus/subset_libero10.json'); print('suite tasks', s['n_tasks']); print('per category', s['counts']); print('name tokens', s['name_tokens']); print('other examples', s['other_examples']); print(len(s['tasks']), 'tasks in subset')"

In [ ]:
# ---- evaluate the final Diffusion Policy checkpoint on the LIBERO-Plus subset ----
import glob, os, subprocess
DRV = '/content/drive/MyDrive/restoring_organs/plus/dp_libero10_ckpts'
last = max(int(d) for d in os.listdir(DRV) if d.isdigit())
pm = glob.glob(f'{DRV}/{last}/**/pretrained_model', recursive=True)
assert pm, f'no pretrained_model under {DRV}/{last}'
print('policy:', pm[0])
subprocess.run(['python', '/content/scripts/lerobot_tools.py', 'eval', '--subset', '/content/drive/MyDrive/restoring_organs/plus/subset_libero10.json', '--out', '/content/drive/MyDrive/restoring_organs/plus/lerobot_dp',
                '--policy', pm[0], '--name', 'diffusion_policy'], check=True, env={**os.environ, 'MUJOCO_GL': 'egl'})